# 실습① — 내 손으로 만드는 고장 예측 모델
**제조 AI 심화과정 1일차 오전 · AI4I 2020 설비 데이터 10,000건 · ㈜에이비에이치**

오늘의 목표 (강의자료 6p): ① 내 과제를 문제 유형으로 정의 ② 고장 예측 모델을 만들어 평가 ③ SHAP으로 판단 근거 설명

**진행 3규칙 (강의자료 35p)** — ① 셀 실행은 `Shift + Enter` ② 위에서 아래로 순서대로 ③ 꼬이면 [런타임]→[세션 다시 시작]→위에서부터 재실행. 막히면 조용히 손을 들어주세요.

> GPU가 필요 없는 실습입니다 — 기본(CPU) 런타임 그대로 진행하세요. (GPU는 오후 이미지 학습에서 사용합니다)

In [ ]:
# [STEP 0] 준비 — 도구를 올리고, 데이터를 데려옵니다 (약 10초)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URLS = [
    "https://raw.githubusercontent.com/abh-edu/mfg-ai-course/main/data/ai4i2020.csv",   # 수업 공식 저장소
    "https://raw.githubusercontent.com/Naveen1402/ai4i2020/main/ai4i2020.csv",          # 예비 경로
]
df = None
for u in URLS:
    try:
        df = pd.read_csv(u)
        break
    except Exception:
        continue
assert df is not None, "데이터 로드 실패 — 네트워크 확인 후 이 셀을 다시 실행하세요."
assert df.shape == (10000, 14), f"예상과 다른 데이터입니다: {df.shape}"
print(f"준비 완료 {df.shape}")

`준비 완료 (10000, 14)` 가 보이면 성공입니다 — **10,000번의 설비 가동 기록, 각 기록마다 14개의 값.**

## STEP 1 — 데이터 첫 대면 (강의자료 40p)
열(컬럼) 14개의 뜻:

| 열 | 뜻 | 비고 |
|---|---|---|
| UDI / Product ID | 일련번호 / 제품 ID | 학습에 쓰면 안 되는 식별자 |
| Type | 제품 등급 (L/M/H) | 저가 50% · 중가 30% · 고가 20% |
| Air/Process temperature [K] | 공기/공정 온도 | 켈빈(K) 단위 |
| Rotational speed [rpm] | 회전수 | |
| Torque [Nm] | 토크(비틀림 힘) | |
| Tool wear [min] | 공구 누적 마모 시간 | |
| **Machine failure** | **고장 여부 (0/1)** | **오늘의 예측 목표** |
| TWF·HDF·PWF·OSF·RNF | 고장 원인 5종 플래그 | ⚠ 사후 기록 — 곧 제거합니다 |

In [ ]:
# 데이터 앞 5행 — 표(DataFrame)의 생김새
df.head()

In [ ]:
# 그래프 한글 준비 (약 20초, 한 번만) — 빨간 경고(warning)가 보여도 정상입니다
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print("한글 그래프 준비 완료")

In [ ]:
# 오늘의 목표 변수: Machine failure — 정상과 고장의 비율은?
cnt = df["Machine failure"].value_counts()
pct = df["Machine failure"].value_counts(normalize=True) * 100
print(f"정상(0): {cnt[0]:,}건 ({pct[0]:.1f}%)")
print(f"고장(1): {cnt[1]:,}건 ({pct[1]:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(["고장 (339건)", "정상 (9,661건)"], [cnt[1], cnt[0]], color=["#C0392B", "#2E7D32"])
for i, v in enumerate([cnt[1], cnt[0]]):
    ax.text(v + 120, i, f"{v:,}건", va="center", fontsize=11)
ax.set_title("함정 ①: 데이터의 96.6%가 정상 — 극단적 불균형", fontsize=13)
ax.set_xlim(0, 11000)
plt.tight_layout(); plt.show()

**여기서 잠깐 (강의자료 16p·40p)** — 모델이 아무 생각 없이 전부 "정상"이라고만 답해도 정확도는 96.6%입니다.
실제 현장은 고장률 0.1% 이하로 더 극단적이죠. 그래서 오늘 우리는 **정확도가 아니라 "고장을 몇 개나 잡았는가(재현율)"** 를 봅니다.

## STEP 2 — 데이터 탐색: 고장은 어디서 일어나는가 (강의자료 41p)

In [ ]:
# 토크(부하) × 공구 마모 — 고장(빨강)이 어디에 모여 있는지 직접 확인
norm = df[df["Machine failure"] == 0]
fail = df[df["Machine failure"] == 1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(norm["Tool wear [min]"], norm["Torque [Nm]"], s=6, alpha=0.25, color="#B0B7C0", label="정상")
ax.scatter(fail["Tool wear [min]"], fail["Torque [Nm]"], s=14, color="#C0392B", label="고장")
ax.axvline(180, ls="--", lw=1, color="#F0641E"); ax.axhline(50, ls="--", lw=1, color="#F0641E")
ax.text(183, ax.get_ylim()[1]*0.95, "마모 180분", color="#F0641E", fontsize=10)
ax.text(2, 51.5, "토크 50Nm", color="#F0641E", fontsize=10)
ax.set_xlabel("공구 마모도 (Tool wear, min)"); ax.set_ylabel("토크 (Torque, Nm)")
ax.set_title("고마모 × 고부하의 교차점에 고장이 군집합니다", fontsize=13)
ax.legend()
plt.tight_layout(); plt.show()

오른쪽 위(마모 큼 + 부하 큼)와 **토크의 위·아래 극단**에 빨간 점이 몰립니다. 단일 변수 임계값 알람이 못 잡는 "조합 패턴" — 이게 AI를 쓰는 이유입니다.

## STEP 3 — 전처리: 정답지를 치우고, 공정하게 나눕니다 (강의자료 42p)

⚠ **함정 ② 데이터 누수 (강의자료 20p)** — `TWF·HDF·PWF·OSF·RNF`는 고장 **원인의 사후 기록**입니다.
실제 예측 시점에는 알 수 없는 값이므로 입력에 넣으면 "정답지를 보고 시험 치는" 셈이 됩니다. 질문 하나면 충분합니다: **"예측 시점에 알 수 있는 값인가?"**

In [ ]:
from sklearn.model_selection import train_test_split

# 1) 식별자 제거(과적합 방지) + 사후 플래그 5종 제거(누수 차단)
X = df.drop(columns=["UDI", "Product ID", "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"])
y = df["Machine failure"]

# 2) 문자 등급(Type: L/M/H) → 숫자 표시로 변환(원핫 인코딩)
X = pd.get_dummies(X, columns=["Type"])
X = X[sorted(X.columns)]  # 열 순서 고정 — 누가 실행해도 같은 결과(재현성)

# 3) 층화 분할 — 학습 75% / 평가 25%, 양쪽 고장 비율을 똑같이 유지
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print(f"학습용: {X_train.shape[0]:,}건 (고장 {int(y_train.sum())}건, {y_train.mean()*100:.2f}%)")
print(f"평가용: {X_test.shape[0]:,}건 (고장 {int(y_test.sum())}건, {y_test.mean()*100:.2f}%)")
print(f"입력 변수 {X.shape[1]}개: {list(X.columns)}")

평가용 2,500건 · 고장 85건 — 이 85건을 몇 개나 잡아내는지가 오늘의 승부입니다.

## STEP 4 — 첫 모델(베이스라인): 정확도의 착시를 직접 봅니다 (강의자료 43p)
랜덤포레스트 — 결정 나무 200그루의 다수결 위원회입니다. 설비 데이터 같은 표 데이터의 표준 출발점이죠.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)          # 학습 — 약 5초
pred = model.predict(X_test)         # 평가용 2,500건 판정

acc = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred)
prec = precision_score(y_test, pred)
f1 = f1_score(y_test, pred)
print(f"정확도 {acc*100:.1f}%  |  고장 재현율 {rec*100:.1f}%  |  고장 정밀도 {prec*100:.1f}%  |  F1 {f1:.3f}")

In [ ]:
# 혼동행렬 — 이 표를 읽을 줄 알면 절반은 끝 (강의자료 17p)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

fig, ax = plt.subplots(figsize=(6.4, 4.6))
mat = [[tn, fp], [fn, tp]]
colors = [["#EAF6EE", "#FFF3CD"], ["#FDEDEB", "#EAF2FF"]]
labels = [[f"TN {tn:,}\n정상→정상", f"FP {fp}\n헛경보"], [f"FN {fn}\n놓친 고장 (위험!)", f"TP {tp}\n고장을 잡음"]]
for r in range(2):
    for c in range(2):
        ax.add_patch(plt.Rectangle((c, 1 - r), 1, 1, facecolor=colors[r][c], edgecolor="#999"))
        ax.text(c + 0.5, 1.5 - r, labels[r][c], ha="center", va="center", fontsize=13,
                color="#C0392B" if (r, c) == (1, 0) else "#333")
ax.set_xticks([0.5, 1.5]); ax.set_xticklabels(["예측: 정상", "예측: 고장"], fontsize=11)
ax.set_yticks([0.5, 1.5]); ax.set_yticklabels(["실제: 고장", "실제: 정상"], fontsize=11)
ax.set_xlim(0, 2); ax.set_ylim(0, 2)
ax.set_title(f"정확도 {acc*100:.1f}% — 그런데 고장 {tp+fn}건 중 {fn}건을 놓쳤습니다", fontsize=13)
plt.tight_layout(); plt.show()
print(f"→ 놓친 고장(FN) {fn}건 = 전체 고장의 {fn/(tp+fn)*100:.1f}% 미감지")

**정확도는 98%대인데, 실제 고장의 30~40%대를 놓치고 있습니다.** 이것이 "정확도의 착시"입니다 —
예지보전에서 놓친 고장(FN) 1건은 돌발 정지·2차 파손으로 직결되는 가장 비싼 실수입니다(강의자료 18p).

> 참고: 여러분 화면의 수치는 강의자료와 소수점 단위로 다를 수 있습니다(라이브러리 버전에 따른 난수 차이). **숫자보다 구조 — "정확도 높음 ≠ 고장 잘 잡음" — 가 핵심입니다.**

## STEP 5 — 불균형 대응 두 처방: Class Weight와 SMOTE (강의자료 19p·44p)

In [ ]:
# 처방 1) Class Weight — "고장을 틀리면 벌점을 크게" (코드 한 줄 추가)
model_cw = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
model_cw.fit(X_train, y_train)
pred_cw = model_cw.predict(X_test)
print(f"[Class Weight] 재현율 {recall_score(y_test, pred_cw)*100:.1f}%  |  정밀도 {precision_score(y_test, pred_cw)*100:.1f}%  |  F1 {f1_score(y_test, pred_cw):.3f}")

In [ ]:
# 처방 2) SMOTE — 고장 데이터 사이를 보간해 합성 사례 생성
# 도구가 없다는 에러(ModuleNotFoundError)가 나면? → 아래처럼 설치하면 됩니다 (강의자료 47p)
!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE

# ⚠ 반드시 "학습 데이터에만" 적용 — 평가 데이터는 원본 그대로! (강의자료 19p)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"학습 데이터: 고장 {int(y_train.sum()):,}건 → {int(y_train_sm.sum()):,}건 (합성으로 균형 확보)")

model_sm = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_sm.fit(X_train_sm, y_train_sm)
pred_sm = model_sm.predict(X_test)
print(f"[SMOTE] 재현율 {recall_score(y_test, pred_sm)*100:.1f}%  |  정밀도 {precision_score(y_test, pred_sm)*100:.1f}%  |  F1 {f1_score(y_test, pred_sm):.3f}")

In [ ]:
# 3전략 비교 — 공짜 점심은 없다 (강의자료 44p)
rows = []
for name, p in [("① 미조치", pred), ("② Class Weight", pred_cw), ("③ SMOTE", pred_sm)]:
    t_n, f_p, f_n, t_p = confusion_matrix(y_test, p).ravel()
    rows.append([name, f"{recall_score(y_test,p)*100:.1f}%", f"{precision_score(y_test,p)*100:.1f}%",
                 f"{f1_score(y_test,p):.3f}", f_n, f_p])
comp = pd.DataFrame(rows, columns=["전략", "재현율(고장 잡음)", "정밀도(경보 신뢰)", "F1", "놓친 고장 FN", "헛경보 FP"])
display(comp)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3); w = 0.35
rec_v = [recall_score(y_test, p)*100 for p in [pred, pred_cw, pred_sm]]
pre_v = [precision_score(y_test, p)*100 for p in [pred, pred_cw, pred_sm]]
ax.bar(x - w/2, rec_v, w, label="고장 재현율", color="#0068FF")
ax.bar(x + w/2, pre_v, w, label="고장 정밀도", color="#203864")
for i in range(3):
    ax.text(x[i] - w/2, rec_v[i] + 1, f"{rec_v[i]:.1f}", ha="center", fontsize=10)
    ax.text(x[i] + w/2, pre_v[i] + 1, f"{pre_v[i]:.1f}", ha="center", fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(["미조치", "Class Weight", "SMOTE"]); ax.set_ylim(0, 105)
ax.set_title("전략별 재현율 vs 정밀도 — 트레이드오프를 눈으로", fontsize=13); ax.legend()
plt.tight_layout(); plt.show()

**읽는 법** — SMOTE는 놓친 고장(FN)을 줄이는 대신 헛경보(FP)를 크게 늘립니다. Class Weight는 랜덤포레스트에서는
효과가 제한적이거나 오히려 떨어지기도 합니다(부트스트랩 표본 안에서 가중치가 희석되는 특성). **만능 처방은 없습니다.**

그래서 선택 기준은 수학이 아니라 **우리 공장의 비용 구조**입니다(강의자료 18p·44p):
- 돌발 정지 피해(미탐 비용) ≫ 점검 출동 비용(오탐 비용) → **SMOTE 계열** (오탐을 감수하고 재현율 확보)
- 오경보 피로·점검 부담이 더 크다 → **미조치 + 임계값 조정** 또는 Class Weight

## STEP 6 — SHAP: "왜 그렇게 판정했는가"를 현장의 언어로 (강의자료 21p·45p)

In [ ]:
# SHAP — 판단 근거를 변수별 기여도로 분해 (설치 약 30초)
!pip install -q shap
import shap

explainer = shap.TreeExplainer(model)              # 베이스라인 모델의 속을 들여다봅니다
sample = X_test.iloc[:500]                          # 속도를 위해 500건 표본
sv = explainer.shap_values(sample)
sv_fail = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)  # "고장" 방향 기여도

shap.summary_plot(sv_fail, sample, show=False, plot_size=(8, 4.5))
plt.title("무엇이 고장 판정을 끌어올리는가 — 변수별 기여도 (붉을수록 값이 큼)", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 개별 사례 해부 — 모델이 "고장"이라 판정한 한 건, 그 이유는?
import numpy as np
idx = np.where((pred == 1) & (y_test.values == 1))[0][0]   # 실제로 잡아낸 고장 1건
one = X_test.iloc[[idx]]
sv_one = explainer.shap_values(one)
sv_one = sv_one[1] if isinstance(sv_one, list) else (sv_one[:, :, 1] if np.array(sv_one).ndim == 3 else sv_one)

contrib = pd.Series(np.ravel(sv_one), index=one.columns).sort_values(key=abs, ascending=False).head(5)
print("이 설비를 '고장 위험'으로 판정한 근거 TOP 5 (+ 위험 방향 / - 안전 방향):")
for k, v in contrib.items():
    print(f"  {'+' if v >= 0 else '-'}{abs(v):.3f}  {k}  (현재값 {one[k].iloc[0]})")
print("\n→ 설비팀에게: \"토크가 높고 마모가 누적된 상태라 위험으로 봤습니다\" — 숫자가 현장의 언어가 됩니다.")

사례에서 보셨듯(강의자료 26p), 이 화면 하나가 "그 변수, 우리도 의심하던 겁니다"라는 반응 — 즉 **현장의 신뢰**를 만듭니다.

## STEP 7 — 마무리: 오늘 완주한 것, 그리고 내 과제로

| 오늘 한 것 | 강의자료 |
|---|---|
| 10,000건 로드 → 불균형 확인 | 40p |
| 탐색(고장의 조합 패턴) | 41p |
| 누수 차단 + 층화 분할 | 42p·20p |
| 베이스라인 → 정확도의 착시 | 43p·17p |
| Class Weight·SMOTE 비교 | 44p·19p |
| SHAP 해석 | 45p·21p |

**보전 관점 3가지 질문 (강의자료 46p)** — 조별로 1분씩 이야기해 보세요:
1. 어떤 전략이 가장 좋은 F1이었고, 왜 그랬는가?
2. 우리 공장이라면 — 미탐 비용과 오탐 비용 중 무엇이 큰가? 그래서 어떤 전략을 고르겠는가?
3. SHAP 상위 변수를 보전팀 점검 항목으로 번역하면 무엇이 되는가?

**실무 확장의 순간 (강의자료 32p)** — STEP 0의 파일 주소만 우리 회사 CSV로 바꾸면, 이 노트북이 그대로 현업 분석의 출발점이 됩니다.

In [ ]:
# ═══════════ [강사 확인용] 슬라이드 이식값 일괄 출력 ═══════════
print("=" * 62)
for name, p in [("베이스라인", pred), ("Class Weight", pred_cw), ("SMOTE", pred_sm)]:
    t_n, f_p, f_n, t_p = confusion_matrix(y_test, p).ravel()
    print(f"[{name:12s}] TN {t_n:4d} | FP {f_p:3d} | FN {f_n:3d} | TP {t_p:3d} | "
          f"acc {accuracy_score(y_test,p)*100:5.1f}% | R {recall_score(y_test,p)*100:5.1f}% | "
          f"P {precision_score(y_test,p)*100:5.1f}% | F1 {f1_score(y_test,p):.3f}")
print("=" * 62)
print("→ 17p·43p·44p 이식용. 리허설(코랩) 실측값을 슬라이드에 반영하십시오.")

---
### 막혔을 때 (강의자료 47p)
1. **런타임 연결 끊김** → 우측 상단 [재연결] → 맨 위 셀부터 순서대로 재실행
2. **ModuleNotFoundError** → 해당 셀 위에 `!pip install 패키지명` 한 줄 추가 실행
3. **결과 수치가 다름** → `random_state=42` 고정 여부 확인 (버전 차이면 소수점 변동은 정상)

---
*데이터 출처: AI4I 2020 Predictive Maintenance Dataset, S. Matzka — UCI Machine Learning Repository (CC BY 4.0). 교육 목적 재배포.*
*ⓒ ㈜에이비에이치 · 제조 AI 심화과정 — 문의: abh@abhcst.com*